# iiq2img — Usage Examples

Fast IIQ-to-image converter for Phase One iXM-GS120 (120MP) raw files with georeferencing support.

In [1]:
from iiq2img import (
    convert_iiq,
    batch_convert,
    extract_metadata,
    extract_geo_info,
)
from pathlib import Path

In [2]:
input_dir = Path("PhaseOneSample")
output_dir = Path().cwd() / "output"

In [3]:
input_files = list(input_dir.glob("*.IIQ"))
first_file = input_files[0]

## Single File Conversion

In [5]:
# Pipeline.LIBRAW  — LibRaw PPG, accurate, ~3s (default)
# Pipeline.FAST    — cv2 EA + numba, ~5× faster, visually equivalent

result = convert_iiq(
    first_file,
    output_dir / (first_file.stem + ".jpg"),
    pipeline="fast",
)

print(f"Output:     {result.output_path}")
print(f"Pipeline:   Pipeline.FAST")
print(f"Resolution: {result.width}x{result.height}")
print(f"Time:       {result.elapsed_ms:.0f}ms")
print(f"Size:       {result.file_size_bytes / 1024 / 1024:.1f} MB")


Output:     /Users/nick/Documents/Projects/iiq2img/output/P0286625.jpg
Pipeline:   .jpg
Resolution: 12768x9564
Time:       1107ms
Size:       45.3 MB


In [10]:
import time
import numpy as np
import cv2
import rawpy
from pathlib import Path
from PIL import Image as PILImage
from turbojpeg import TurboJPEG

from iiq2img.converter import (
    extract_metadata,
    _read_iiq_exif_and_xmp,
    _inject_metadata_into_jpeg,
)

iiq = first_file
_turbo = TurboJPEG()


def T(label, fn):
    t0 = time.perf_counter()
    result = fn()
    ms = (time.perf_counter() - t0) * 1000
    print(f"{label:<35} {ms:>7.1f} ms")
    return result


print(f"{'Step':<35} {'Time':>10}")
print("-" * 47)

# 1. Metadata extraction
metadata = T("extract_metadata (EXIF/XMP)", lambda: extract_metadata(iiq))


# 2. Raw decode + demosaic (LibRaw PPG)
def demosaic():
    raw = rawpy.imread(str(iiq))
    rgb = raw.postprocess(
        demosaic_algorithm=rawpy.DemosaicAlgorithm.PPG,
        half_size=False,
        use_camera_wb=True,
        output_color=rawpy.ColorSpace.sRGB,
        output_bps=8,
    )
    raw.close()
    return rgb


rgb = T("demosaic (LibRaw PPG)", demosaic)

# 3. RGB -> BGR color conversion
bgr = T("cvtColor RGB->BGR", lambda: cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))


# 4a. JPEG encode via OpenCV
def encode_cv2():
    ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
    return buf.tobytes()


cv2_bytes = T("cv2.imencode JPEG q=90", encode_cv2)

# 4b. JPEG encode via TurboJPEG
turbo_bytes = T("TurboJPEG.encode q=90", lambda: _turbo.encode(bgr, quality=90))

# 5. Write bytes to disk
out_cv2 = Path("/tmp/bench_cv2.jpg")
out_turbo = Path("/tmp/bench_turbo.jpg")
T("write cv2 bytes to disk", lambda: out_cv2.write_bytes(cv2_bytes))
T("write turbo bytes to disk", lambda: out_turbo.write_bytes(turbo_bytes))

# 6. Metadata injection (EXIF/XMP re-write)
import shutil

shutil.copy(out_turbo, "/tmp/bench_meta.jpg")
exif_bytes, xmp_bytes = _read_iiq_exif_and_xmp(iiq)
T(
    "inject EXIF+XMP into JPEG",
    lambda: _inject_metadata_into_jpeg("/tmp/bench_meta.jpg", exif_bytes, xmp_bytes),
)

print()
print(f"cv2 output:   {len(cv2_bytes) / 1024 / 1024:.1f} MB")
print(f"turbo output: {len(turbo_bytes) / 1024 / 1024:.1f} MB")


Step                                      Time
-----------------------------------------------
extract_metadata (EXIF/XMP)             0.5 ms
demosaic (LibRaw PPG)                3101.6 ms
cvtColor RGB->BGR                      43.7 ms
cv2.imencode JPEG q=90                389.1 ms
TurboJPEG.encode q=90                 433.8 ms
write cv2 bytes to disk                15.6 ms
write turbo bytes to disk              13.7 ms
inject EXIF+XMP into JPEG              39.3 ms

cv2 output:   55.9 MB
turbo output: 62.3 MB


## Bulk Conversion

In [ ]:
results = batch_convert(
    input_dir=input_dir,
    output_dir=output_dir,
    output_format="jpg",
    compress_quality=90,
    workers=8,
    pipeline="fast",
)


  0%|          | 0/70 [00:00<?, ?img/s]

## Georeferencing

Adds spatial reference using GPS/IMU data from IIQ metadata.
- **JPEG/PNG**: sidecar world file (.jgw/.pgw)
- **TIFF**: embedded GeoTIFF with CRS and affine transform

In [ ]:
# JPEG with world file
result = convert_iiq(first_file, output_dir / (first_file.stem + ".jpg"), georef=True)

In [ ]:
# GeoTIFF with embedded CRS
result = convert_iiq(first_file, output_dir / (first_file.stem + ".tif"), georef=True)

In [ ]:
# GSD and footprint from metadata
meta = extract_metadata("PhaseOneSample/P0286625.IIQ")
geo = extract_geo_info(
    meta["xmp"], focal_length_mm=80.0, image_width=12768, image_height=9564
)

print(f"Position:  {geo.latitude:.6f}, {geo.longitude:.6f}")
print(f"Altitude:  {geo.altitude_agl:.1f}m AGL")
print(f"GSD:       {geo.gsd * 100:.2f} cm/pixel")
print(f"Footprint: {geo.footprint_width:.1f}m x {geo.footprint_height:.1f}m")

Position:  -34.560007, 118.378565
Altitude:  25.8m AGL
GSD:       0.11 cm/pixel
Footprint: 14.2m x 10.6m
